# MSH chain on a 2D substrate: dense and sparse Python reference

Reference for `msh_substrate_tn.ipynb`: the same model as a sparse matrix (SciPy), with dense diagonalization where it fits and sparse
methods beyond. See the tensor-network notebook for the goal and the model.

**Contents**
1. Hamiltonian
2. Check against the tensor-network Hamiltonian
3. Lowest levels against substrate width
4. Sparse matrix size against system size
5. Zero-mode benchmark: dense and sparse

**Conventions.** Site index $n = i_x + i_y N_x$ (as in the Julia notebook), matrix index $4n + c$ with kwant's spinor
$(c_\uparrow, c_\downarrow, c^\dagger_\downarrow, -c^\dagger_\uparrow)$, so the BdG Hamiltonian reads
$\tau_z(\epsilon - \mu) + \Delta\tau_x + J\,\mathbf S\cdot\boldsymbol\sigma$ on site, with Rashba hoppings $h_x = -t\tau_z + i\alpha\tau_z\sigma_y$,
$h_y = -t\tau_z - i\alpha\tau_z\sigma_x$ ($H_{r+d,\,r} = h_d$).

In [1]:
import os, time, resource, queue
import multiprocessing as mp
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[v] = "1"        # single thread (set before NumPy is imported), as in the Julia notebook
import numpy as np
import scipy.linalg as la
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh, splu, LinearOperator

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

s0 = np.eye(2); sx = np.array([[0, 1], [1, 0]]); sy = np.array([[0, -1j], [1j, 0]]); sz = np.diag([1.0, -1.0])
tzs0, txs0 = np.kron(sz, s0), np.kron(sx, s0)
t0sx, t0sy, t0sz = np.kron(s0, sx), np.kron(s0, sy), np.kron(s0, sz)
tzsx, tzsy = np.kron(sz, sx), np.kron(sz, sy)

S_z = lambda ix: (0.0, 0.0, 1.0)
S_tilt = lambda ix: (np.sin(0.3*ix)*np.cos(0.7), np.sin(0.3*ix)*np.sin(0.7), np.cos(0.3*ix))

## 1. Hamiltonian

`ham(Lx, Ly, ell, S)` builds the sparse BdG matrix from Kronecker products: with $n = i_y N_x + i_x$, a site operator is
`kron(A_y, B_x)`, and the full matrix is `kron(site operator, 4×4 spinor matrix)`. The x bonds are `kron(I_y, shift_x)` (open, so
no wrap between rows), the y bonds `kron(shift_y, I_x)`. The exchange is `kron(diag(J S_k), σ_k)` on the chain sites.

In [2]:
def shift(n):
    "n × n matrix with ones at (i + 1, i): |i⟩ → |i + 1⟩, open boundary"
    return sp.diags(np.ones(n - 1), -1, shape=(n, n), format="csr")

def chain_sites(Lx, Ly, ell):
    Nx, Ny = 2**Lx, 2**Ly
    x0, iy0 = (Nx - ell) // 2, Ny // 2
    return np.arange(x0, x0 + ell), iy0

def ham(Lx, Ly, ell, S=S_z, t=1.0, mu=-7.0, alpha=0.9, Delta=2.4, J=5.8):
    "Sparse 4N × 4N BdG Hamiltonian (CSR) of the MSH chain on the 2^Lx × 2^Ly substrate."
    Nx, Ny = 2**Lx, 2**Ly; N = Nx * Ny
    Ix, Iy = sp.identity(Nx, format="csr"), sp.identity(Ny, format="csr")
    hx = -t * tzs0 + 1j * alpha * tzsy
    hy = -t * tzs0 - 1j * alpha * tzsx
    Tx = sp.kron(sp.kron(Iy, shift(Nx)), hx)
    Ty = sp.kron(sp.kron(shift(Ny), Ix), hy)
    H = sp.kron(sp.identity(N), -mu * tzs0 + Delta * txs0) + Tx + Tx.conj().T + Ty + Ty.conj().T
    xs, iy0 = chain_sites(Lx, Ly, ell)
    for k, sk in enumerate((t0sx, t0sy, t0sz)):
        v = np.zeros(N)
        v[xs + iy0 * Nx] = [J * S(ix)[k] for ix in xs]
        if np.abs(v).max() > 1e-14:
            H = H + sp.kron(sp.diags(v), sk)
    return H.tocsr()

## 2. Check against the tensor-network Hamiltonian

`data/tn_check_H.npz` holds the dense MPO matrices of `msh_substrate_tn.ipynb` (section 4), with Julia's index $n_b\cdot2N + \sigma N + n$
and spinor $(c_\uparrow, c_\downarrow, c^\dagger_\uparrow, c^\dagger_\downarrow)$. The change of basis to $4n + c$ with
$(c_\uparrow, c_\downarrow, c^\dagger_\downarrow, -c^\dagger_\uparrow)$ is a signed permutation $W$; the check is
$\max|W H_{\rm TN} W^T - H_{\rm py}|$ element by element.

In [3]:
def julia_to_python(Hj, N):
    "Julia basis (index nb·2N + σ·N + n) → Python basis (index 4n + c)."
    W = sp.lil_matrix((4 * N, 4 * N))
    for n in range(N):
        W[4*n + 0, 0*N + n] = 1            # c↑   ← (nb = 0, σ = ↑)
        W[4*n + 1, 1*N + n] = 1            # c↓   ← (nb = 0, σ = ↓)
        W[4*n + 2, 2*N + 1*N + n] = 1      # c†↓  ← (nb = 1, σ = ↓)
        W[4*n + 3, 2*N + 0*N + n] = -1     # −c†↑ ← (nb = 1, σ = ↑)
    W = W.tocsr()
    return W @ Hj @ W.T

tn_check = np.load(os.path.join(DATA_DIR, "tn_check_H.npz"))
worst = 0.0
for key in sorted(tn_check.files):
    _, Lx, Ly, ell, name = key.split("_"); Lx, Ly, ell = int(Lx), int(Ly), int(ell)
    Hp = ham(Lx, Ly, ell, S_z if name == "z" else S_tilt).toarray()
    err = np.abs(julia_to_python(tn_check[key], 2**(Lx + Ly)) - Hp).max(); worst = max(worst, err)
    print(f"Lx = {Lx}, Ly = {Ly}, ℓ = {ell:2d}, {name:<4s}: D = {Hp.shape[0]:3d}, max|H_TN − H_py| = {err:.1e}")
print(f"largest difference: {worst:.1e}")
assert worst < 1e-10

Lx = 2, Ly = 3, ℓ =  3, tilt: D = 128, max|H_TN − H_py| = 4.1e-14
Lx = 2, Ly = 3, ℓ =  3, z   : D = 128, max|H_TN − H_py| = 2.4e-14
Lx = 3, Ly = 0, ℓ =  6, tilt: D =  32, max|H_TN − H_py| = 4.7e-14
Lx = 3, Ly = 0, ℓ =  6, z   : D =  32, max|H_TN − H_py| = 3.0e-14
Lx = 3, Ly = 1, ℓ =  6, tilt: D =  64, max|H_TN − H_py| = 1.9e-14
Lx = 3, Ly = 1, ℓ =  6, z   : D =  64, max|H_TN − H_py| = 2.8e-14
Lx = 3, Ly = 2, ℓ =  5, tilt: D = 128, max|H_TN − H_py| = 9.3e-14
Lx = 3, Ly = 2, ℓ =  5, z   : D = 128, max|H_TN − H_py| = 1.8e-14
Lx = 4, Ly = 2, ℓ = 10, tilt: D = 256, max|H_TN − H_py| = 1.3e-13
Lx = 4, Ly = 2, ℓ = 10, z   : D = 256, max|H_TN − H_py| = 6.1e-14
largest difference: 1.3e-13


## 3. Lowest levels against substrate width

$L_x = 6$ (64 columns), chain $\ell = 48$, all spins along $\hat z$; the four smallest $|E|$ from shift-invert `eigsh` ($\sigma = 0$).
The first pair is the Majorana pair; the next level is the bulk gap. $L_y = 0$ is the 1D chain. Because the substrate at $\mu = -7$ is
gapped, the gap converges after a few rows: from there on, growing the substrate changes the cost but not the physics, which makes
the converged values a correctness check at sizes where dense diagonalization is no longer possible.

In [4]:
print(f"{'Ly':>3s} {'rows':>5s} {'D':>7s} {'|E| pair':>10s} {'gap':>8s} {'eigsh':>8s}")
for Ly in range(7):
    H = ham(6, Ly, 48)
    t_ = time.perf_counter()
    E = np.sort(np.abs(eigsh(H, k=6, sigma=0, which="LM", return_eigenvectors=False)))
    print(f"{Ly:3d} {2**Ly:5d} {H.shape[0]:7d} {E[0]:10.2e} {E[2]:8.4f} {time.perf_counter() - t_:7.2f}s")

 Ly  rows       D   |E| pair      gap    eigsh
  0     1     256   2.60e-11   0.2563    0.00s
  1     2     512   5.10e-15   0.4283    0.01s
  2     4    1024   6.74e-14   0.5613    0.08s
  3     8    2048   3.40e-14   0.5491    0.13s
  4    16    4096   3.39e-14   0.5491    0.35s
  5    32    8192   3.39e-14   0.5491    1.31s
  6    64   16384   3.39e-14   0.5491    4.05s


## 4. Sparse matrix size against system size

The width and square scans of section 5 of the tensor-network notebook: number of nonzeros of the sparse BdG matrix, its CSR memory
and the build time. The MPO memory (stored elements of its tensors) is listed there for comparison.

In [5]:
scan = [(6, Ly, 48) for Ly in range(7)] + [(L, L, 3 * 2**L // 4) for L in range(4, 9)]
print(f"{'Lx':>3s} {'Ly':>3s} {'sites':>7s} {'D':>8s} {'nnz':>10s} {'CSR MB':>8s} {'build':>8s}")
for Lx, Ly, ell in scan:
    t_ = time.perf_counter(); H = ham(Lx, Ly, ell); t_build = time.perf_counter() - t_
    mb = (H.data.nbytes + H.indices.nbytes + H.indptr.nbytes) / 1e6
    print(f"{Lx:3d} {Ly:3d} {2**(Lx+Ly):7d} {H.shape[0]:8d} {H.nnz:10d} {mb:8.2f} {t_build:7.3f}s")

 Lx  Ly   sites        D        nnz   CSR MB    build
  6   0      64      256       3040     0.06   0.001s
  6   1     128      512       8128     0.16   0.001s
  6   2     256     1024      18304     0.37   0.001s
  6   3     512     2048      38656     0.78   0.001s
  6   4    1024     4096      79360     1.60   0.002s
  6   5    2048     8192     160768     3.25   0.002s
  6   6    4096    16384     323584     6.54   0.004s
  4   4     256     1024      19456     0.39   0.001s
  5   5    1024     4096      79872     1.61   0.001s
  6   6    4096    16384     323584     6.54   0.003s
  7   7   16384    65536    1302528    26.31   0.020s
  8   8   65536   262144    5226496   105.58   0.095s


## 5. Zero-mode benchmark: dense and sparse

The same sizes as section 7 of the tensor-network notebook ($\ell = 12$, spins along $\hat z$, up to $1024\times1024$ sites), single
thread:

- **Dense** (while $D \le$ `DENSE_MAX_D` $= 16\,384$, i.e. up to $64\times64$ sites): `scipy.linalg.eigh` of the full matrix,
  restricted to the four eigenpairs in the middle of the spectrum (`subset_by_index`), which is what one does for the zero modes
  (about twice as fast as the full spectrum; the cost is the $O(D^3)$ reduction to tridiagonal form either way).
- **Sparse**: shift-invert Lanczos, `eigsh(H, k=4, sigma=0, OPinv=…)`, i.e. Lanczos on $H^{-1}$ with a sparse LU factorization of
  $H$ (SuperLU, `splu`). Its cost and memory are dominated by the fill-in of the $L$ and $U$ factors, which grows faster than $N$ on
  a 2D lattice, so the fill-reducing ordering matters. `eigsh` alone uses SuperLU's default column ordering (COLAMD); the matrix is
  structurally symmetric, and the symmetric minimum-degree ordering `MMD_AT_PLUS_A` halves the fill and makes the solve 2.3–2.5 times
  faster (measured at $256\times256$: 21 s → 9 s; $512\times512$: 276 s → 110 s). The benchmark therefore factorizes with
  `splu(H, permc_spec="MMD_AT_PLUS_A")` and passes the solve to `eigsh` as `OPinv`; the time includes the factorization.

  At $1024\times1024$ sites ($D = 4\,194\,304$) SuperLU stops with *"Not enough memory to perform factorization"* within about a second
  under either ordering, with only a few GB in use on a machine with 188 GB: the factorization exceeds SuperLU's limits, not the
  machine's memory. That size is recorded as failed. (A different sparse direct solver, e.g. MUMPS or PARDISO with nested-dissection
  ordering, might go further; it is not part of this benchmark.)

**Memory.** Each (size, method) runs in a fresh process forked from this kernel (`in_fresh_process`), which reports its peak resident
memory (`ru_maxrss`). The forked process starts with the kernel's resident memory, which is printed once as the baseline. The time
includes only the eigensolver (not building $H$). If the calculation raises an error or the process dies, the row is recorded with
`status` set to the error and NaN values.

From the four eigenpairs: $E_f$ (the $+E$ member of the pair), $\Delta Q = \langle\psi_f|\tau_z|\psi_f\rangle$ of its eigenvector, and
the gap (third smallest $|E|$). Each row is appended to `py_zero_modes.csv` as soon as it is done, and sizes already in that file are
skipped, so an interrupted run can be resumed (delete the file to start over). `RUN_ZM = False` only loads it.

In [6]:
RUN_ZM = True
DENSE_MAX_D = 16384
zm_file = "py_zero_modes.csv"
zm_scan = sorted([(6, Ly) for Ly in range(7)] + [(L, L) for L in (4, 5, 7, 8, 9, 10)], key=lambda p: (sum(p), p[1]))  # increasing N

def pair_values(E, V):
    "E_f, ΔQ of the +E member of the pair, and the gap, from a few eigenpairs around zero."
    o = np.argsort(np.abs(E)); ip = o[:2][np.argmax(E[o[:2]])]
    v = V[:, ip].reshape(-1, 4)
    return E[ip], np.sum(np.abs(v[:, :2])**2) - np.sum(np.abs(v[:, 2:])**2), np.abs(E[o[2]])

def solve(Lx, Ly, method, ell=12):
    "Zero modes of one system by one method: (time of the eigensolver, E_f, ΔQ, gap)."
    H = ham(Lx, Ly, ell); N = H.shape[0] // 4
    if method == "sparse":
        H = H.tocsc()
        t_ = time.perf_counter()
        lu = splu(H, permc_spec="MMD_AT_PLUS_A")
        E, V = eigsh(H, k=4, sigma=0, which="LM", OPinv=LinearOperator(H.shape, matvec=lu.solve, dtype=complex))
    else:
        H = H.toarray()
        t_ = time.perf_counter(); E, V = la.eigh(H, subset_by_index=[2*N - 2, 2*N + 1])
    return (time.perf_counter() - t_, *pair_values(E, V))

def in_fresh_process(fn, *args):
    "Run fn(*args) in a forked process. Returns (result or None, peak resident memory in MB, status)."
    ctx = mp.get_context("fork"); q = ctx.Queue()
    def target():
        try:
            out, status = fn(*args), "ok"
        except BaseException as e:
            out, status = None, f"{type(e).__name__}: {e}".replace(",", ";").strip()
        q.put((out, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, status))
    p = ctx.Process(target=target); p.start()
    while True:                                          # wait for the result, but notice a process that dies (e.g. killed)
        try:
            res = q.get(timeout=5); break
        except queue.Empty:
            if not p.is_alive():
                res = (None, float("nan"), f"process died (exit code {p.exitcode})"); break
    p.join()
    return res

cols = ["Lx", "Ly", "ell", "N", "D", "method", "t", "rss_MB", "E_f", "dQ", "E_gap", "status"]
done = set()
if os.path.isfile(zm_file):
    old = np.genfromtxt(zm_file, delimiter=",", names=True, dtype=None, encoding=None, ndmin=1)
    done = {(int(r["Lx"]), int(r["Ly"]), r["method"]) for r in old}
if RUN_ZM:
    solve(4, 0, "sparse"); solve(4, 0, "dense")                       # warm-up
    print(f"baseline peak memory of a forked process: {in_fresh_process(lambda: None)[1]:.0f} MB")
    if not done:
        with open(zm_file, "w") as f:
            f.write(",".join(cols) + "\n")
    for Lx, Ly in zm_scan:
        D = 4 * 2**(Lx + Ly)
        for method in ("sparse", "dense") if D <= DENSE_MAX_D else ("sparse",):
            if (Lx, Ly, method) in done:
                continue
            out, rss, status = in_fresh_process(solve, Lx, Ly, method)
            t, Ef, dQ, gap = out if out is not None else (np.nan,) * 4
            with open(zm_file, "a") as f:
                f.write(",".join(str(x) for x in (Lx, Ly, 12, D // 4, D, method, t, rss, Ef, dQ, gap, status)) + "\n")
            print(f"done Lx = {Lx}, Ly = {Ly}, {method}: {t:.2f} s, {rss:.0f} MB, {status}", flush=True)

zm = np.genfromtxt(zm_file, delimiter=",", names=True, dtype=None, encoding=None, ndmin=1)
zm = zm[np.lexsort((zm["method"], zm["N"]))]
print(f"{'Lx':>3s} {'Ly':>3s} {'N':>8s} {'D':>9s} {'method':>7s} {'time':>9s} {'RSS MB':>9s} {'E_f':>13s} {'ΔQ':>13s} {'gap':>7s}  status")
for r in zm:
    print(f"{r['Lx']:3d} {r['Ly']:3d} {r['N']:8d} {r['D']:9d} {r['method']:>7s} {r['t']:8.2f}s {r['rss_MB']:9.0f} {r['E_f']:13.6e} "
          f"{r['dQ']:+13.6e} {r['E_gap']:7.4f}  {r['status']}")

baseline peak memory of a forked process: 276 MB
done Lx = 6, Ly = 0, sparse: 0.00 s, 286 MB, ok
done Lx = 6, Ly = 0, dense: 0.01 s, 286 MB, ok
done Lx = 6, Ly = 1, sparse: 0.00 s, 286 MB, ok
done Lx = 6, Ly = 1, dense: 0.03 s, 286 MB, ok
done Lx = 6, Ly = 2, sparse: 0.01 s, 286 MB, ok
done Lx = 6, Ly = 2, dense: 0.18 s, 286 MB, ok
done Lx = 4, Ly = 4, sparse: 0.01 s, 286 MB, ok
done Lx = 4, Ly = 4, dense: 0.18 s, 286 MB, ok
done Lx = 6, Ly = 3, sparse: 0.01 s, 287 MB, ok
done Lx = 6, Ly = 3, dense: 1.51 s, 350 MB, ok
done Lx = 6, Ly = 4, sparse: 0.04 s, 286 MB, ok
done Lx = 6, Ly = 4, dense: 13.47 s, 590 MB, ok
done Lx = 5, Ly = 5, sparse: 0.04 s, 288 MB, ok
done Lx = 5, Ly = 5, dense: 13.62 s, 575 MB, ok
done Lx = 6, Ly = 5, sparse: 0.09 s, 301 MB, ok
done Lx = 6, Ly = 5, dense: 110.53 s, 1408 MB, ok
done Lx = 6, Ly = 6, sparse: 0.25 s, 324 MB, ok
done Lx = 6, Ly = 6, dense: 879.82 s, 4581 MB, ok
done Lx = 7, Ly = 7, sparse: 1.64 s, 479 MB, ok
done Lx = 8, Ly = 8, sparse: 9.53 s, 141